# 02. Data Cleaning & Validation Pipeline

**Project**: Cardiovascular Disease Data Analysis  
**Goal**: Audit data quality, enforce physiological validation rules, perform feature engineering, and export `cardio_cleaned.csv`.

---

## 1. Overview & Validation Rules
We enforce five strict physiological domain boundaries:
1. `height`: `100 <= height <= 220` cm
2. `weight`: `30 <= weight <= 200` kg
3. `ap_hi` (Systolic BP): `60 <= ap_hi <= 240` mmHg
4. `ap_lo` (Diastolic BP): `40 <= ap_lo <= 160` mmHg
5. `ap_hi > ap_lo` (Systolic BP must strictly exceed Diastolic BP)

> [!IMPORTANT]
> The records failing these rules represent data entry errors / invalid records (e.g. 16000 mmHg BP), NOT statistical outliers.


In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

from src.data_loader import load_raw_data, save_to_sqlite
from src.data_cleaning import perform_cleaning
from src.feature_engineering import add_engineered_features

df_raw = load_raw_data('../data/raw/cardio_train.csv')
df_clean, audit = perform_cleaning(df_raw)

audit_df = pd.DataFrame([audit]).T.reset_index()
audit_df.columns = ['Metric', 'Value']
audit_df

,Metric,Value
0,total_raw_records,70000.00
1,missing_records,0.00
2,duplicate_records,0.00
3,invalid_height_records,30.00
4,invalid_weight_records,7.00
5,invalid_systolic_bp_records,228.00
6,invalid_diastolic_bp_records,1025.00
7,invalid_bp_relationship_records,1236.00
8,multiple_rule_violations,1155.00
9,final_excluded_records,1364.00


## 2. Feature Engineering

In [2]:
df_feat = add_engineered_features(df_clean)
print("Engineered DataFrame Shape:", df_feat.shape)
df_feat[['age_years', 'age_group', 'gender_label', 'BMI', 'BMI_Category', 'Blood_Pressure_Category']].head()

Engineered DataFrame Shape: (68636, 25)


,age_years,age_group,gender_label,BMI,BMI_Category,Blood_Pressure_Category
0,50.4,50-59,Male,21.97,Normal Weight,Hypertension Stage 1
1,55.4,50-59,Female,34.93,Obese Class I,Hypertension Stage 2
2,51.6,50-59,Female,23.51,Normal Weight,Hypertension Stage 1
3,48.2,40-49,Male,28.71,Overweight,Hypertension Stage 2
4,47.8,40-49,Female,23.01,Normal Weight,Normal


## 3. Export Cleaned Dataset

In [3]:
df_feat.to_csv('../data/processed/cardio_cleaned.csv', index=False)
save_to_sqlite(df_feat, '../data/processed/cardio_db.sqlite')
print("Successfully exported cleaned CSV and SQLite database.")

Data saved to SQLite database: ../data/processed/cardio_db.sqlite (Table: cardio_cleaned)
Successfully exported cleaned CSV and SQLite database.
